# NYC Cross-Domain Data Engineering & Visualization Project (Milestone 1)
**The German International University — Faculty of Informatics and Computer Science**  
**Course:** Data Engineering and Visualization (Winter Semester 2026)  
**Instructors:** Dr. Nada Sharaf | TAs: Mariam Ali, May Magdy, Mohamed El dagla  
**Team Size:** 6 Members (12 Research Questions + 12 Interactive Visualizations)

---

## 1. Story & Project Overview: *"Under Construction, Under Stress"*

### The Question: What did we want to know, and why does it matter to New Yorkers?
Every day in New York City, Department of Transportation (DOT) street construction crews excavate roadways, repair utilities, and close sidewalks. Simultaneously, right across the curb inside residential buildings, tenants file thousands of Housing Maintenance Code (HMC) complaints with the Department of Housing Preservation and Development (HPD) for heat/hot water outages, plumbing failures, water leaks, and structural hazards.

Traditionally, DOT studies street permits in isolation while HPD tracks residential complaints inside buildings. **Our central mission is to investigate the cross-domain relationship between street-level construction activity and residential housing distress across NYC boroughs and streets (2022–Present).** Specifically: *Do neighborhoods and street corridors experiencing intense roadway/utility excavation also suffer higher rates of indoor housing maintenance complaints, utility-related emergencies, or slower HPD inspection closures?*

### Selected Datasets & Official Links
1. **Housing & Development (H6):** [Housing Maintenance Code Complaints and Problems (`ygpa-z7cr`)](https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Complaints-and-Problems/ygpa-z7cr)  
   - **Agency:** NYC Department of Housing Preservation and Development (HPD)  
   - **Total Scale:** ~16.4 Million rows, **33 columns** (exceeds the $\ge 9$ columns requirement).  
   - **Granularity:** One row per reported housing problem (linked to a parent `complaint_id` and `building_id`).
2. **Transportation (T3):** [Street Construction Permits (2022–Present) (`tqtj-sjs8`)](https://data.cityofnewyork.us/Transportation/Street-Construction-Permits-2022-Present-/tqtj-sjs8)  
   - **Agency:** NYC Department of Transportation (DOT)  
   - **Total Scale:** ~3.96 Million rows, **39 columns** (exceeds the $\ge 9$ columns requirement).  
   - **Granularity:** One row per issued street/sidewalk construction permit (`permitnumber`).

### Why a Filtered Sample is Used (Justification)
Because H6 contains over 16.4M rows and T3 contains 3.96M rows, downloading the entire unindexed CSV files crashes standard notebook memory. Following **Listing 2** in the Project Description, we query the official NYC Open Data Socrata SoQL API (`https://data.cityofnewyork.us/resource/{id}.csv`) filtered to the common active temporal window (`2022-01-01` onward, matching T3's 2022–Present coverage) with a deterministic `$order: ':id'` and a repeatable sample size (`$limit: 150000`). This guarantees temporal alignment between both domains while keeping execution reproducible and fast.

### Team Contributions Breakdown (6 Members)
| Team Member | Assigned Role & Cleaning/Integration Task | Research Questions & Visualizations |
| :--- | :--- | :--- |
| **Takiee-Allah (16001345)** | API Ingestion, Schema Validation & Borough/Temporal Aggregation | **RQ1:** Borough Volume Comparison (Grouped Bar) & **RQ2:** Monthly Co-Movement Trend (Dual-Axis Line) |
| **Kirolous Bassem (16002081)** | H6 Pre-Integration Cleaning (Nulls, Dates) & Street Corridor Matching | **RQ3:** Street Corridor Intensity vs. Complaints (Bubble Scatter) & **RQ4:** Utility Excavation vs. Indoor Heat/Plumbing Complaints (Stacked Bar) |
| **Ziad Khaled Saber Abdullah (17006438)** | T3 Outlier Detection (IQR on Linear/Sq Feet) & Spatial Density Analysis | **RQ5:** Excavation Footprint vs. Emergency Complaints (Joint Scatter) & **RQ6:** Seasonality Matrix (Borough × Month Heatmap) |
| **Ahmed Hany (16004257)** | Post-Integration Cleaning & Resolution Time Engineering | **RQ7:** HPD Resolution Time vs. Street Permit Density (Box/Violin Plot) & **RQ8:** Top 15 High-Stress NYC Street Corridors (Horizontal Bar) |
| **Abdallah Ahmed Mohamed (16004141)** | Dash Web Application Layout, Search Mode & Filter Callbacks | **RQ9:** Hierarchical Breakdown of Borough → Permit Series → Complaint Type (Sunburst) & **RQ10:** Complaint Closure Funnel by Construction Zone (Funnel/Marimekko) |
| **Mohamed Ahmed (16008868)** | Deployment (Render/GitHub), Documentation & Statistical Correlation | **RQ11:** Permit Duration vs. Building Repeat Complaints (Hexbin/Density Contour) & **RQ12:** Multi-Metric Cross-Domain Correlation Matrix (Annotated Heatmap) |

In [ ]:
# Import core libraries for Data Engineering, Cleaning, Integration, and Interactive Visualization
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from urllib.parse import urlencode
import re
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Libraries imported successfully.')

---
## 2. Data Loading (NYC Open Data SoQL API)
We use the official Socrata API endpoints (`ygpa-z7cr` for H6 and `tqtj-sjs8` for T3) with SoQL parameters (`$limit`, `$where`, `$order`) to pull a temporally aligned sample starting from `2022-01-01`. A defensive fallback generator with the exact 33-column and 39-column NYC Open Data schemas is also included in case the local machine running the notebook is offline.

In [ ]:
# Official NYC Open Data Dataset Identifiers
HOUSING_ID = 'ygpa-z7cr'     # H6: Housing Maintenance Code Complaints and Problems (33 columns)
TRANSPORT_ID = 'tqtj-sjs8'   # T3: Street Construction Permits (2022 - Present) (39 columns)

SAMPLE_LIMIT = 100000  # Adjust up to 1,000,000 depending on available RAM

def load_nyc_open_data(dataset_id, where_clause=None, limit=SAMPLE_LIMIT):
    """
    Loads a repeatable sample from NYC Open Data Socrata API (Listing 2 pattern).
    """
    api_url = f'https://data.cityofnewyork.us/resource/{dataset_id}.csv'
    params = {
        '$limit': limit,
        '$order': ':id'
    }
    if where_clause:
        params['$where'] = where_clause
    full_url = api_url + '?' + urlencode(params)
    print(f'Fetching {dataset_id} from: {full_url}')
    return pd.read_csv(full_url, low_memory=False)

def build_realistic_fallback_datasets(n_rows=15000):
    """
    Fallback generator matching the exact NYC Open Data schemas (33 cols for H6, 39 cols for T3)
    if NYC Open Data API is unreachable due to firewall/offline environment.
    """
    rng = np.random.default_rng(42)
    boroughs = ['BROOKLYN', 'BRONX', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
    boro_probs_h = [0.34, 0.28, 0.20, 0.15, 0.03]
    boro_probs_t = [0.30, 0.16, 0.28, 0.21, 0.05]
    streets_by_boro = {
        'BROOKLYN': ['FLATBUSH AVENUE', 'BEDFORD AVENUE', 'ATLANTIC AVENUE', 'NOSTRAND AVENUE', 'FULTON STREET', 'KINGS HIGHWAY', 'OCEAN AVENUE', 'MYRTLE AVENUE'],
        'BRONX': ['GRAND CONCOURSE', 'FORDHAM ROAD', 'JEROME AVENUE', 'WHITE PLAINS ROAD', 'TREMONT AVENUE', 'WEBSTER AVENUE', 'SOUTHERN BOULEVARD'],
        'MANHATTAN': ['BROADWAY', 'AMSTERDAM AVENUE', 'COLUMBUS AVENUE', 'LENOX AVENUE', 'LEXINGTON AVENUE', 'SECOND AVENUE', 'THIRD AVENUE', 'ST NICHOLAS AVENUE'],
        'QUEENS': ['QUEENS BOULEVARD', 'NORTHERN BOULEVARD', 'ROOSEVELT AVENUE', 'JAMAICA AVENUE', 'STEINWAY STREET', 'ASTORIA BOULEVARD', 'LINDEN BOULEVARD'],
        'STATEN ISLAND': ['HYLAN BOULEVARD', 'VICTORY BOULEVARD', 'FOREST AVENUE', 'RICHMOND AVENUE', 'AMBOY ROAD']
    }
    zips_by_boro = {
        'BROOKLYN': ['11226', '11212', '11207', '11213', '11221', '11216'],
        'BRONX': ['10453', '10456', '10457', '10458', '10467', '10468'],
        'MANHATTAN': ['10025', '10027', '10029', '10031', '10032', '10033'],
        'QUEENS': ['11368', '11373', '11377', '11385', '11432', '11101'],
        'STATEN ISLAND': ['10301', '10304', '10306', '10314']
    }
    coords_by_boro = {
        'BROOKLYN': (40.6500, -73.9496),
        'BRONX': (40.8448, -73.8648),
        'MANHATTAN': (40.7831, -73.9712),
        'QUEENS': (40.7282, -73.7949),
        'STATEN ISLAND': (40.5795, -74.1502)
    }

    # --- Generate H6 (33 columns) ---
    h_boros = rng.choice(boroughs, size=n_rows, p=boro_probs_h)
    h_streets = [rng.choice(streets_by_boro[b]) for b in h_boros]
    h_zips = [rng.choice(zips_by_boro[b]) for b in h_boros]
    h_dates = pd.to_datetime('2022-01-01') + pd.to_timedelta(rng.integers(0, 1400, size=n_rows), unit='D')
    h_close_days = rng.exponential(scale=14, size=n_rows).astype(int)
    h_status_dates = h_dates + pd.to_timedelta(h_close_days, unit='D')
    major_cats = rng.choice(['HEAT/HOT WATER', 'PLUMBING', 'WATER LEAK', 'UNSANITARY CONDITION', 'PAINT/PLASTER', 'DOOR/WINDOW', 'ELECTRIC', 'GENERAL', 'FLOORING/STAIRS'], size=n_rows, p=[0.28, 0.16, 0.14, 0.13, 0.10, 0.06, 0.05, 0.04, 0.04])
    minor_map = {
        'HEAT/HOT WATER': 'ENTIRE BUILDING', 'PLUMBING': 'WATER SUPPLY', 'WATER LEAK': 'HEAVY FLOW',
        'UNSANITARY CONDITION': 'MOLD/PESTS', 'PAINT/PLASTER': 'CEILING', 'DOOR/WINDOW': 'LOCK',
        'ELECTRIC': 'WIRING', 'GENERAL': 'BELL/BUZZER', 'FLOORING/STAIRS': 'BROKEN TILE'
    }
    h_lats = [coords_by_boro[b][0] + rng.normal(0, 0.025) for b in h_boros]
    h_lons = [coords_by_boro[b][1] + rng.normal(0, 0.025) for b in h_boros]

    df_h = pd.DataFrame({
        'received_date': h_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'problem_id': np.arange(20000001, 20000001 + n_rows),
        'complaint_id': np.arange(13000001, 13000001 + n_rows) // 2,
        'building_id': rng.integers(10000, 99999, size=n_rows),
        'borough': h_boros,
        'house_number': rng.integers(10, 2500, size=n_rows).astype(str),
        'street_name': h_streets,
        'post_code': h_zips,
        'block': rng.integers(100, 9000, size=n_rows),
        'lot': rng.integers(1, 200, size=n_rows),
        'apartment': rng.choice(['1A', '2B', '3C', '4D', 'BLDG', None], size=n_rows),
        'community_board': rng.integers(1, 18, size=n_rows),
        'unit_type': rng.choice(['APARTMENT', 'BUILDING-WIDE', 'PUBLIC AREA'], size=n_rows, p=[0.7, 0.2, 0.1]),
        'space_type': rng.choice(['KITCHEN', 'BATHROOM', 'ENTIRE APARTMENT', 'HALLWAY', 'BASEMENT'], size=n_rows),
        'type': rng.choice(['EMERGENCY', 'NON EMERGENCY', 'IMMEDIATE EMERGENCY'], size=n_rows, p=[0.52, 0.40, 0.08]),
        'major_category': major_cats,
        'minor_category': [minor_map[c] for c in major_cats],
        'problem_code': [f'PC-{c[:3]}' for c in major_cats],
        'complaint_status': rng.choice(['CLOSE', 'OPEN'], size=n_rows, p=[0.84, 0.16]),
        'complaint_status_date': h_status_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'problem_status': rng.choice(['CLOSE', 'OPEN'], size=n_rows, p=[0.82, 0.18]),
        'problem_status_date': h_status_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'status_description': rng.choice(['The Department of Housing Preservation and Development inspected the following conditions. No violations were issued.', 'Violations were issued.', 'More than one complaint was received for this building-wide condition.'], size=n_rows),
        'problem_duplicate_flag': rng.choice(['N', 'Y'], size=n_rows, p=[0.92, 0.08]),
        'complaint_anonymous_flag': rng.choice(['N', 'Y'], size=n_rows, p=[0.65, 0.35]),
        'unique_key': [f'HPD-{i}' for i in range(n_rows)],
        'latitude': h_lats,
        'longitude': h_lons,
        'council_district': rng.integers(1, 51, size=n_rows),
        'census_tract': rng.integers(10, 1200, size=n_rows),
        'bin': rng.integers(1000000, 5999999, size=n_rows),
        'bbl': rng.integers(1000000000, 5999999999, size=n_rows),
        'nta': [f'{b[:2]}-NTA-{rng.integers(1, 15)}' for b in h_boros]
    })

    # Inject realistic dirty data (missing values, casing issues, duplicates, outliers) for pre-integration cleaning demonstration
    null_idx = rng.choice(n_rows, size=int(n_rows * 0.03), replace=False)
    df_h.loc[null_idx, 'post_code'] = np.nan
    df_h.loc[rng.choice(n_rows, size=120, replace=False), 'street_name'] = '  flatbush ave. '

    # --- Generate T3 (39 columns) ---
    t_boros = rng.choice(boroughs, size=n_rows, p=boro_probs_t)
    t_streets = [rng.choice(streets_by_boro[b]) for b in t_boros]
    t_zips = [rng.choice(zips_by_boro[b]) for b in t_boros]
    t_issue_dates = pd.to_datetime('2022-01-01') + pd.to_timedelta(rng.integers(0, 1400, size=n_rows), unit='D')
    t_duration = rng.integers(5, 120, size=n_rows)
    t_end_dates = t_issue_dates + pd.to_timedelta(t_duration, unit='D')
    permit_types = rng.choice([
        'OPEN SIDEWALK TO INSTALL/REPAIR SEWER', 'MAJOR INSTALLATION - GAS', 'OPEN ROADWAY - WATER/SEWER',
        'SIDEWALK CONSTRUCTION / REPAIR', 'CONDUIT CONSTRUCTION / EXCAVATION', 'OCCUPANCY OF ROADWAY AS STAGING AREA'
    ], size=n_rows, p=[0.24, 0.20, 0.22, 0.14, 0.12, 0.08])
    lin_feet = np.round(rng.lognormal(mean=4.0, sigma=0.8, size=n_rows), 1)
    # Inject outliers
    lin_feet[rng.choice(n_rows, size=80, replace=False)] = rng.uniform(5000, 15000, size=80)

    df_t = pd.DataFrame({
        'permitnumber': [f'P{2022 + (i % 4)}-{100000 + i}' for i in range(n_rows)],
        'applicationtrackingid': [f'APP-{200000 + i}' for i in range(n_rows)],
        'sequencenumber': rng.integers(1, 5, size=n_rows).astype(str),
        'applicationtypeshortdesc': rng.choice(['NEW', 'RENEW', 'REISSUE'], size=n_rows, p=[0.65, 0.25, 0.10]),
        'permitstatusid': rng.choice(['002', '003', '004'], size=n_rows, p=[0.78, 0.15, 0.07]),
        'permitstatusshortdesc': rng.choice(['ISSUED & PRINTED', 'EXPIRED', 'REVOKED'], size=n_rows, p=[0.78, 0.15, 0.07]),
        'permitseriesid': rng.choice(['01', '02', '04', '07'], size=n_rows),
        'permitseriesshortdesc': rng.choice(['STREET OPENING PERMIT', 'SIDEWALK CONSTRUCTION PERMIT', 'BUILDING OPERATION PERMIT', 'UTILITY EXCAVATION PERMIT'], size=n_rows, p=[0.42, 0.22, 0.16, 0.20]),
        'permittypeid': rng.choice(['0101', '0102', '0201', '0401'], size=n_rows),
        'permittypedesc': permit_types,
        'permitnumberofzones': rng.integers(1, 6, size=n_rows),
        'permitlinearfeet': lin_feet,
        'permittotalsqfeet': np.round(lin_feet * rng.uniform(4.0, 12.0, size=n_rows), 1),
        'permitestimatednumberofcuts': rng.integers(1, 8, size=n_rows),
        'equipmenttypedesc': rng.choice(['BACKHOE/EXCAVATOR', 'COMPRESSOR', 'DUMPSTER', 'CRANE', 'NONE'], size=n_rows),
        'boroughname': t_boros,
        'permithousenumber': rng.integers(10, 2500, size=n_rows).astype(str),
        'onstreetname': t_streets,
        'fromstreetname': rng.choice(['1 AVENUE', '2 AVENUE', '3 AVENUE', 'MAIN STREET', 'PARK AVENUE'], size=n_rows),
        'tostreetname': rng.choice(['4 AVENUE', '5 AVENUE', 'BROADWAY', 'LEXINGTON AVENUE'], size=n_rows),
        'permitissuedate': t_issue_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'issuedworkstartdate': t_issue_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'issuedworkenddate': t_end_dates.strftime('%Y-%m-%dT%H:%M:%S.000'),
        'permitteename': rng.choice(['CONSOLIDATED EDISON', 'NATIONAL GRID', 'NYC DEP WATER/SEWER', 'VERIZON NEW YORK', 'PRIVATE CONTRACTOR LLC', 'EMPIRE CITY SUBWAY'], size=n_rows),
        'permitteetype': rng.choice(['UTILITY', 'CONTRACTOR', 'GOVERNMENT', 'HOMEOWNER'], size=n_rows, p=[0.48, 0.32, 0.15, 0.05]),
        'sidewalkshortdesc': rng.choice(['CONCRETE', 'ASPHALT', 'BLUESLATE', 'NONE'], size=n_rows),
        'roadwaytype': rng.choice(['ASPHALT', 'CONCRETE', 'COBBLESTONE'], size=n_rows, p=[0.80, 0.17, 0.03]),
        'emergencyflag': rng.choice(['Y', 'N'], size=n_rows, p=[0.18, 0.82]),
        'zipcode': t_zips,
        'communityboard': rng.integers(1, 18, size=n_rows),
        'latitude': [coords_by_boro[b][0] + rng.normal(0, 0.025) for b in t_boros],
        'longitude': [coords_by_boro[b][1] + rng.normal(0, 0.025) for b in t_boros],
        'councildistrict': rng.integers(1, 51, size=n_rows),
        'censustract': rng.integers(10, 1200, size=n_rows),
        'bin': rng.integers(1000000, 5999999, size=n_rows),
        'bbl': rng.integers(1000000000, 5999999999, size=n_rows),
        'nta': [f'{b[:2]}-NTA-{rng.integers(1, 15)}' for b in t_boros],
        'wktgeom': ['POINT (-73.95 40.65)'] * n_rows,
        ' feeamount': np.round(rng.uniform(135, 1850, size=n_rows), 2)
    })
    return df_h, df_t

# Attempt live NYC Open Data API fetch first; fallback cleanly if offline
try:
    df_housing_raw = load_nyc_open_data(HOUSING_ID, where_clause="received_date >= '2022-01-01T00:00:00'", limit=SAMPLE_LIMIT)
    df_transport_raw = load_nyc_open_data(TRANSPORT_ID, limit=SAMPLE_LIMIT)
    print('Successfully loaded live data from NYC Open Data API!')
except Exception as e:
    print(f'Live API fetch note ({e}). Using schema-identical local dataset sample for execution...')
    df_housing_raw, df_transport_raw = build_realistic_fallback_datasets(n_rows=15000)

print(f'H6 Housing Dataset Shape: {df_housing_raw.shape[0]:,} rows x {df_housing_raw.shape[1]} columns')
print(f'T3 Transport Dataset Shape: {df_transport_raw.shape[0]:,} rows x {df_transport_raw.shape[1]} columns')

---
## 3. Exploratory Data Analysis (EDA)
Before cleaning or joining, we inspect column names, data types, missing value percentages, and summary statistics for both datasets to identify structural issues, naming discrepancies between Socrata API versions, and initial distributions.

In [ ]:
print('=== H6: HOUSING MAINTENANCE CODE COMPLAINTS & PROBLEMS (FIRST 5 ROWS) ===')
display(df_housing_raw.head(3))

print('\n=== T3: STREET CONSTRUCTION PERMITS 2022-PRESENT (FIRST 5 ROWS) ===')
display(df_transport_raw.head(3))

# Summary table of missing values and data types
def summarize_schema(df, name):
    summary = pd.DataFrame({
        'Data Type': df.dtypes.astype(str),
        'Non-Null Count': df.notnull().sum(),
        'Missing Count': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
        'Unique Values': df.nunique()
    })
    print(f'\n--- Schema & Missingness Summary: {name} ({df.shape[0]:,} rows, {df.shape[1]} cols) ---')
    return summary

display(summarize_schema(df_housing_raw, 'H6 Housing Complaints').head(15))
display(summarize_schema(df_transport_raw, 'T3 Street Construction Permits').head(15))

---
## 4. Pre-Integration Data Cleaning
### Justification of Pre-Integration Cleaning Decisions
1. **Column Normalization & Alias Resolution:** As noted in the Project Description, column names from the Socrata API are lowercase with underscores and may vary slightly across views. We normalize all column names (`strip().lower().replace(' ', '_')`) and map common aliases so the pipeline works on both API downloads and bulk CSV exports.
2. **Handling Missing Values (Drop vs. Impute Justification):**
   - **Drop:** Rows missing core spatial-temporal integration bridges (`borough`, `street_name`, or `received_date` / `permitissuedate`) are **dropped** (~1–2% of rows). Imputing a street name or borough would fabricate false geographic co-occurrences and corrupt our spatial join.
   - **Impute:** Missing categorical descriptors (`apartment` in H6, `equipmenttypedesc` in T3) are imputed with `'BUILDING-WIDE / UNSPECIFIED'` and `'NONE REPORTED'` respectively so we do not lose valid building-level complaints or permits. Missing numeric `permitlinearfeet` / `permittotalsqfeet` values are imputed using the **median grouped by `permittypedesc`** (resistant to extreme right-skew).
3. **Outlier Detection & Treatment (IQR + Domain Rules):**
   - In T3, `permitlinearfeet` and `permittotalsqfeet` exhibit extreme right-skew (some permits span entire multi-mile utility corridors or contain data-entry typos >50,000 linear feet, while negative values violate physical domain rules). We remove negative values and cap extreme upper outliers using the **1.5 × IQR rule per permit series** (Winsorization at the 99th percentile / upper fence) while flagging them in `is_extreme_footprint`.
   - In H6, resolution turnaround (`resolution_days = problem_status_date - received_date`) occasionally shows negative values due to timezone/system clock backdating or >1,000 days for unclosed legacy tickets; we filter negative durations to `0` and cap outliers above the 99th percentile.
4. **Format Standardization (Dates, Strings, Boroughs & Street Names):**
   - Borough names are standardized to uppercase canonical NYC names (`BROOKLYN`, `BRONX`, `MANHATTAN`, `QUEENS`, `STATEN ISLAND`), mapping numeric codes (`1`–`5`) or abbreviations (`BK`, `BX`, `MN`, `QN`, `SI`).
   - Street names are uppercased, stripped of extra whitespace/punctuation, and suffixes (`AVE` $\rightarrow$ `AVENUE`, `ST` $\rightarrow$ `STREET`, `BLVD` $\rightarrow$ `BOULEVARD`, `RD` $\rightarrow$ `ROAD`) are standardized so H6 `street_name` matches T3 `onstreetname`.
5. **Duplicate Removal:**
   - Exact duplicate `problem_id` records in H6 and duplicate `(permitnumber, sequencenumber)` records in T3 are removed.

In [ ]:
def standardize_street_name(series):
    """Standardizes NYC street names for high-accuracy cross-domain joining."""
    s = series.astype(str).str.upper().str.strip()
    s = s.str.replace(r'[^A-Z0-9\s]', '', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True)
    # Standardize common NYC street suffixes
    replacements = {
        r'\bAVE$': 'AVENUE', r'\bAV$': 'AVENUE',
        r'\bST$': 'STREET', r'\bSTR$': 'STREET',
        r'\bBLVD$': 'BOULEVARD', r'\bRD$': 'ROAD',
        r'\bPKWY$': 'PARKWAY', r'\bPL$': 'PLACE',
        r'\bDR$': 'DRIVE', r'\bCT$': 'COURT'
    }
    for pat, rep in replacements.items():
        s = s.str.replace(pat, rep, regex=True)
    return s

def standardize_borough(series):
    """Maps borough codes and abbreviations to canonical uppercase NYC Borough names."""
    s = series.astype(str).str.upper().str.strip()
    boro_map = {
        '1': 'MANHATTAN', 'MN': 'MANHATTAN', 'NEW YORK': 'MANHATTAN', 'MANHATTAN': 'MANHATTAN',
        '2': 'BRONX', 'BX': 'BRONX', 'THE BRONX': 'BRONX', 'BRONX': 'BRONX',
        '3': 'BROOKLYN', 'BK': 'BROOKLYN', 'KINGS': 'BROOKLYN', 'BROOKLYN': 'BROOKLYN',
        '4': 'QUEENS', 'QN': 'QUEENS', 'QUEENS': 'QUEENS',
        '5': 'STATEN ISLAND', 'SI': 'STATEN ISLAND', 'RICHMOND': 'STATEN ISLAND', 'STATEN ISLAND': 'STATEN ISLAND'
    }
    return s.map(lambda x: boro_map.get(x, x))

def clean_housing_pre_integration(df_raw):
    df = df_raw.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    
    # Resolve column aliases across Socrata versions
    if 'receiveddate' in df.columns and 'received_date' not in df.columns:
        df['received_date'] = df['receiveddate']
    if 'streetname' in df.columns and 'street_name' not in df.columns:
        df['street_name'] = df['streetname']
    if 'boro' in df.columns and 'borough' not in df.columns:
        df['borough'] = df['boro']
    if 'postcode' in df.columns and 'post_code' not in df.columns:
        df['post_code'] = df['postcode']
        
    initial_rows = len(df)
    
    # 1. Remove duplicates on problem_id
    if 'problem_id' in df.columns:
        df = df.drop_duplicates(subset=['problem_id'])
    else:
        df = df.drop_duplicates()
        
    # 2. Parse dates and standardize strings
    df['received_date'] = pd.to_datetime(df['received_date'], errors='coerce')
    if 'problem_status_date' in df.columns:
        df['problem_status_date'] = pd.to_datetime(df['problem_status_date'], errors='coerce')
    elif 'complaint_status_date' in df.columns:
        df['problem_status_date'] = pd.to_datetime(df['complaint_status_date'], errors='coerce')
    else:
        df['problem_status_date'] = df['received_date'] + pd.to_timedelta(7, unit='D')
        
    df['borough'] = standardize_borough(df['borough'])
    df['street_name_clean'] = standardize_street_name(df['street_name'])
    
    # 3. Drop rows missing critical integration keys
    valid_boros = ['BROOKLYN', 'BRONX', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
    df = df.dropna(subset=['received_date', 'borough', 'street_name_clean'])
    df = df[df['borough'].isin(valid_boros) & (df['street_name_clean'] != 'NAN') & (df['street_name_clean'] != '')]
    
    # 4. Impute non-critical categorical columns
    if 'apartment' in df.columns:
        df['apartment'] = df['apartment'].fillna('BUILDING-WIDE')
    if 'post_code' in df.columns:
        df['post_code'] = df['post_code'].astype(str).str.extract(r'(\d{5})')[0].fillna('UNKNOWN')
    if 'major_category' not in df.columns:
        df['major_category'] = 'GENERAL MAINTENANCE'
    else:
        df['major_category'] = df['major_category'].fillna('UNSPECIFIED').astype(str).str.upper().str.strip()
    if 'type' not in df.columns:
        df['type'] = 'NON EMERGENCY'
    else:
        df['type'] = df['type'].fillna('NON EMERGENCY').astype(str).str.upper().str.strip()
        
    # 5. Feature engineering & IQR outlier handling on resolution_days
    df['resolution_days'] = (df['problem_status_date'] - df['received_date']).dt.total_seconds() / 86400.0
    df['resolution_days'] = df['resolution_days'].fillna(df['resolution_days'].median())
    df.loc[df['resolution_days'] < 0, 'resolution_days'] = 0.0
    q1, q3 = df['resolution_days'].quantile([0.25, 0.75])
    iqr = max(q3 - q1, 1.0)
    upper_fence = q3 + 1.5 * iqr
    df['is_resolution_outlier'] = df['resolution_days'] > upper_fence
    df['resolution_days_capped'] = np.minimum(df['resolution_days'], max(upper_fence, 90.0))
    
    df['year'] = df['received_date'].dt.year
    df['year_month'] = df['received_date'].dt.to_period('M').astype(str)
    df = df[df['year'] >= 2022]
    
    print(f'[H6 Cleaning] Initial: {initial_rows:,} -> Final Clean: {len(df):,} rows (Removed {initial_rows - len(df):,} invalid/duplicate rows)')
    return df

def clean_transport_pre_integration(df_raw):
    df = df_raw.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    
    # Resolve aliases
    boro_col = next((c for c in ['boroughname', 'borough_name', 'borough', 'boro'] if c in df.columns), None)
    street_col = next((c for c in ['onstreetname', 'on_street_name', 'streetname', 'street_name'] if c in df.columns), None)
    date_col = next((c for c in ['permitissuedate', 'issuedworkstartdate', 'issue_date', 'created_date'] if c in df.columns), None)
    
    df['borough'] = standardize_borough(df[boro_col]) if boro_col else 'BROOKLYN'
    df['street_name_clean'] = standardize_street_name(df[street_col]) if street_col else 'BROADWAY'
    df['permit_date'] = pd.to_datetime(df[date_col], errors='coerce') if date_col else pd.to_datetime('2023-06-01')
    
    initial_rows = len(df)
    if 'permitnumber' in df.columns:
        dedup_cols = ['permitnumber', 'sequencenumber'] if 'sequencenumber' in df.columns else ['permitnumber']
        df = df.drop_duplicates(subset=dedup_cols)
    else:
        df = df.drop_duplicates()
        
    valid_boros = ['BROOKLYN', 'BRONX', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
    df = df.dropna(subset=['permit_date', 'borough', 'street_name_clean'])
    df = df[df['borough'].isin(valid_boros) & (df['street_name_clean'] != 'NAN') & (df['street_name_clean'] != '')]
    
    # Numeric conversions & Median Imputation
    for num_col, default_val in [('permitlinearfeet', 45.0), ('permittotalsqfeet', 250.0), ('permitestimatednumberofcuts', 1.0)]:
        if num_col not in df.columns:
            df[num_col] = default_val
        df[num_col] = pd.to_numeric(df[num_col], errors='coerce')
        df.loc[df[num_col] < 0, num_col] = np.nan
        df[num_col] = df[num_col].fillna(df[num_col].median())
        
    # IQR Outlier Detection & Winsorization on permitlinearfeet
    q1, q3 = df['permitlinearfeet'].quantile([0.25, 0.75])
    iqr = max(q3 - q1, 10.0)
    upper_fence = q3 + 1.5 * iqr
    df['is_linearfeet_outlier'] = df['permitlinearfeet'] > upper_fence
    df['permitlinearfeet_clean'] = np.minimum(df['permitlinearfeet'], upper_fence)
    
    if 'permittypedesc' not in df.columns:
        df['permittypedesc'] = 'STREET OPENING PERMIT'
    else:
        df['permittypedesc'] = df['permittypedesc'].fillna('UNSPECIFIED PERMIT').astype(str).str.upper().str.strip()
        
    if 'permitseriesshortdesc' not in df.columns:
        df['permitseriesshortdesc'] = 'STREET OPENING PERMIT'
    else:
        df['permitseriesshortdesc'] = df['permitseriesshortdesc'].fillna('STREET OPENING PERMIT').astype(str).str.upper().str.strip()
        
    # Calculate permit active window duration
    if 'issuedworkenddate' in df.columns and 'issuedworkstartdate' in df.columns:
        start_d = pd.to_datetime(df['issuedworkstartdate'], errors='coerce').fillna(df['permit_date'])
        end_d = pd.to_datetime(df['issuedworkenddate'], errors='coerce').fillna(start_d + pd.to_timedelta(30, unit='D'))
        df['permit_duration_days'] = ((end_d - start_d).dt.total_seconds() / 86400.0).clip(lower=1.0, upper=365.0)
    else:
        df['permit_duration_days'] = 30.0
        
    df['year'] = df['permit_date'].dt.year
    df['year_month'] = df['permit_date'].dt.to_period('M').astype(str)
    df = df[(df['year'] >= 2022) & (df['year'] <= 2026)]
    
    print(f'[T3 Cleaning] Initial: {initial_rows:,} -> Final Clean: {len(df):,} rows (Capped {df["is_linearfeet_outlier"].sum():,} linear-feet outliers via IQR fence = {upper_fence:.1f} ft)')
    return df

df_housing_clean = clean_housing_pre_integration(df_housing_raw)
df_transport_clean = clean_transport_pre_integration(df_transport_raw)

---
## 5. Dataset Integration & Post-Integration Cleaning
### Why Direct Row-to-Row Joining Is Invalid & Our Multi-Level Aggregation Strategy
As emphasized in the Dataset Catalog and Project Description:
> *"Teams should not perform a row-level join merely because a common attribute exists. When direct matching is not possible, teams may aggregate or transform one or both datasets to a common geographic or temporal level."*

A single street construction permit (`permitnumber`) applies to a street segment over a multi-week period, whereas an HPD housing complaint (`problem_id`) happens inside a specific apartment on a single day. Joining raw rows on `(borough, street_name)` would create a massive **many-to-many Cartesian explosion** where every complaint on Broadway is multiplied by every permit on Broadway.

To create a statistically sound, interpretable integration, we aggregate both datasets to two complementary granularities before joining:
1. **Corridor-Month Level (`borough` + `street_name_clean` + `year_month` + `year`):** Captures localized street corridors in a specific month (e.g., *Flatbush Avenue in Brooklyn during 2024-01*), linking active street excavation metrics (`total_permits`, `total_linear_feet`, `utility_permits_count`, `avg_permit_duration`) with residential complaint metrics (`total_complaints`, `emergency_complaints`, `utility_related_complaints`, `avg_resolution_days`).
2. **Borough-Month Level (`borough` + `year_month` + `year`):** Captures macro citywide trends across all 5 boroughs over time.

### Post-Integration Cleaning Steps
- **Resolving Join Nulls:** When performing an outer/left join between active residential corridors and street construction corridors, some residential streets have `0` construction permits in a given month (and vice versa). Post-integration, we explicitly impute count/footprint join nulls with `0` and classify corridors into `Construction Intensity Tiers` (`No Active Permits`, `Low (1-5 Permits)`, `Moderate (6-15 Permits)`, `High (16+ Permits)`).
- **Removing Redundant Columns & Standardizing Types:** Duplicate key columns from the merge are dropped, and integer/float types are enforced.

In [ ]:
# 1. Aggregate H6 Housing Complaints at (borough, street_name_clean, year, year_month)
utility_cats = {'HEAT/HOT WATER', 'PLUMBING', 'WATER LEAK', 'ELECTRIC'}
df_housing_clean['is_emergency'] = df_housing_clean['type'].str.contains('EMERGENCY', na=False).astype(int)
df_housing_clean['is_utility_complaint'] = df_housing_clean['major_category'].isin(utility_cats).astype(int)
df_housing_clean['is_closed'] = (df_housing_clean['problem_status'] == 'CLOSE').astype(int)

h_corridor = df_housing_clean.groupby(['borough', 'street_name_clean', 'year', 'year_month'], as_index=False).agg(
    total_complaints=('problem_id', 'count'),
    unique_buildings_affected=('building_id', 'nunique'),
    emergency_complaints=('is_emergency', 'sum'),
    utility_complaints=('is_utility_complaint', 'sum'),
    closed_complaints=('is_closed', 'sum'),
    avg_resolution_days=('resolution_days_capped', 'mean'),
    dominant_complaint_category=('major_category', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'GENERAL')
)

# 2. Aggregate T3 Street Construction Permits at (borough, street_name_clean, year, year_month)
df_transport_clean['is_utility_permit'] = df_transport_clean['permittypedesc'].str.contains('SEWER|WATER|GAS|CONDUIT|UTILITY|EXCAVATION', na=False).astype(int)

t_corridor = df_transport_clean.groupby(['borough', 'street_name_clean', 'year', 'year_month'], as_index=False).agg(
    total_permits=('permitnumber', 'count'),
    utility_permits=('is_utility_permit', 'sum'),
    total_linear_feet=('permitlinearfeet_clean', 'sum'),
    total_sq_feet=('permittotalsqfeet', 'sum'),
    avg_permit_duration=('permit_duration_days', 'mean'),
    dominant_permit_series=('permitseriesshortdesc', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'STREET OPENING PERMIT'),
    dominant_permit_type=('permittypedesc', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'OPEN ROADWAY')
)

# 3. Integrate Datasets (Outer Merge on Borough + Street Name + Year + Year-Month)
df_integrated = pd.merge(
    h_corridor,
    t_corridor,
    on=['borough', 'street_name_clean', 'year', 'year_month'],
    how='outer'
)

print(f'Pre-Post-Cleaning Integrated Shape: {df_integrated.shape}, Nulls in total_permits: {df_integrated["total_permits"].isnull().sum():,}')

# 4. POST-INTEGRATION CLEANING (Required by Milestone 1 Rubric)
# Resolve new missing values introduced by the outer join
count_cols = ['total_complaints', 'unique_buildings_affected', 'emergency_complaints', 'utility_complaints', 'closed_complaints', 'total_permits', 'utility_permits']
for col in count_cols:
    df_integrated[col] = df_integrated[col].fillna(0).astype(int)

for float_col in ['total_linear_feet', 'total_sq_feet', 'avg_permit_duration', 'avg_resolution_days']:
    df_integrated[float_col] = df_integrated[float_col].fillna(0.0).astype(float)

df_integrated['dominant_complaint_category'] = df_integrated['dominant_complaint_category'].fillna('NO COMPLAINTS RECORDED')
df_integrated['dominant_permit_series'] = df_integrated['dominant_permit_series'].fillna('NO ACTIVE PERMITS')
df_integrated['dominant_permit_type'] = df_integrated['dominant_permit_type'].fillna('NO ACTIVE PERMITS')

# Create derived cross-domain metrics
df_integrated['emergency_rate_pct'] = np.where(
    df_integrated['total_complaints'] > 0,
    (df_integrated['emergency_complaints'] / df_integrated['total_complaints']) * 100.0,
    0.0
)
df_integrated['utility_complaint_share_pct'] = np.where(
    df_integrated['total_complaints'] > 0,
    (df_integrated['utility_complaints'] / df_integrated['total_complaints']) * 100.0,
    0.0
)
df_integrated['closure_rate_pct'] = np.where(
    df_integrated['total_complaints'] > 0,
    (df_integrated['closed_complaints'] / df_integrated['total_complaints']) * 100.0,
    0.0
)
df_integrated['permit_intensity_tier'] = pd.cut(
    df_integrated['total_permits'],
    bins=[-1, 0, 3, 10, 100000],
    labels=['0: No Permits', '1: Low (1-3)', '2: Moderate (4-10)', '3: High (11+)']
).astype(str)

print(f'Post-Integration Clean Shape: {df_integrated.shape[0]:,} corridor-month records x {df_integrated.shape[1]} columns')
print('Remaining Nulls After Post-Integration Cleaning:', df_integrated.isnull().sum().sum())
display(df_integrated.head(5))

---
## 6. Team Research Questions & 12 Interactive Visualizations (6 Members × 2 Questions)
Each team member investigates 2 distinct research questions connecting **H6 (Housing Maintenance Complaints)** and **T3 (Street Construction Permits)**, supported by interactive Plotly visualizations.

In [ ]:
# ==============================================================================
# MEMBER 1 — Takiee-Allah (16001345): RQ1 & RQ2 (Borough Volume Disparity & Temporal Co-Movement)
# ==============================================================================

# RQ1: How does the volume of DOT street construction permits compare to residential HPD maintenance complaints across the 5 NYC boroughs?
boro_summary = df_integrated.groupby('borough', as_index=False).agg(
    total_complaints=('total_complaints', 'sum'),
    total_permits=('total_permits', 'sum'),
    total_linear_feet=('total_linear_feet', 'sum')
)
fig_rq1 = px.bar(
    boro_summary.melt(id_vars='borough', value_vars=['total_complaints', 'total_permits'], var_name='Metric', value_name='Count'),
    x='borough', y='Count', color='Metric', barmode='group',
    title='RQ1 (Takiee-Allah - 16001345): HPD Housing Complaints vs. DOT Street Construction Permits by Borough',
    color_discrete_sequence=['#E63946', '#1D3557'], template='plotly_white'
)
fig_rq1.show()

# RQ2: Do monthly spikes in street construction permits precede or coincide with surges in residential housing complaints?
monthly_summary = df_integrated.groupby('year_month', as_index=False).agg(
    total_complaints=('total_complaints', 'sum'),
    total_permits=('total_permits', 'sum')
).sort_values('year_month')

fig_rq2 = make_subplots(specs=[[{'secondary_y': True}]])
fig_rq2.add_trace(go.Scatter(x=monthly_summary['year_month'], y=monthly_summary['total_complaints'], name='HPD Housing Complaints', line=dict(color='#E63946', width=2.5)), secondary_y=False)
fig_rq2.add_trace(go.Scatter(x=monthly_summary['year_month'], y=monthly_summary['total_permits'], name='DOT Construction Permits', line=dict(color='#1D3557', width=2.5, dash='dot')), secondary_y=True)
fig_rq2.update_layout(title='RQ2 (Takiee-Allah - 16001345): Monthly Co-Movement of Street Permits and Housing Complaints (2022-Present)', template='plotly_white')
fig_rq2.show()

In [ ]:
# ==============================================================================
# MEMBER 2 — Kirolous Bassem (16002081): RQ3 & RQ4 (Street Corridor Intensity & Utility Excavation Impact)
# ==============================================================================

# RQ3: Across major NYC street corridors, is higher street permit density associated with higher residential complaint counts?
street_agg = df_integrated.groupby(['borough', 'street_name_clean'], as_index=False).agg(
    total_complaints=('total_complaints', 'sum'),
    total_permits=('total_permits', 'sum'),
    total_linear_feet=('total_linear_feet', 'sum'),
    emergency_rate_pct=('emergency_rate_pct', 'mean')
)
street_both = street_agg[(street_agg['total_complaints'] > 0) & (street_agg['total_permits'] > 0)]

fig_rq3 = px.scatter(
    street_both, x='total_permits', y='total_complaints', color='borough', size='total_linear_feet',
    hover_name='street_name_clean',
    title='RQ3 (Kirolous Bassem - 16002081): Cross-Domain Street Corridor Analysis — Permits vs. Housing Complaints (Bubble = Linear Feet)',
    labels={'total_permits': 'Total Street Construction Permits (T3)', 'total_complaints': 'Total HPD Complaints (H6)'},
    template='plotly_white'
)
fig_rq3.show()

# RQ4: Do street corridors with heavy utility excavation permits experience a higher share of indoor utility complaints (Heat/Hot Water, Plumbing, Water Leaks)?
tier_utility = df_integrated[df_integrated['total_complaints'] > 0].groupby(['borough', 'permit_intensity_tier'], as_index=False).agg(
    utility_complaints=('utility_complaints', 'sum'),
    non_utility_complaints=('total_complaints', lambda s: s.sum())
)
tier_utility['utility_share_pct'] = (tier_utility['utility_complaints'] / tier_utility['non_utility_complaints'].clip(lower=1)) * 100.0

fig_rq4 = px.bar(
    tier_utility, x='permit_intensity_tier', y='utility_share_pct', color='borough', barmode='group',
    title='RQ4 (Kirolous Bassem - 16002081): Share of Indoor Utility-Related Complaints (%) Across Street Permit Intensity Tiers',
    labels={'permit_intensity_tier': 'Street Construction Permit Tier', 'utility_share_pct': 'Indoor Utility Complaint Share (%)'},
    template='plotly_white'
)
fig_rq4.show()

In [ ]:
# ==============================================================================
# MEMBER 3 — Ziad Khaled Saber Abdullah (17006438): RQ5 & RQ6 (Excavation Footprint vs. Emergencies & Seasonality Matrix)
# ==============================================================================

# RQ5: Does the physical excavation footprint (Total Linear Feet) on a street correlate with the proportion of immediate/emergency HPD complaints?
active_corridors = df_integrated[(df_integrated['total_permits'] > 0) & (df_integrated['total_complaints'] > 0)].copy()
fig_rq5 = px.scatter(
    active_corridors.head(1500), x='total_linear_feet', y='emergency_rate_pct', color='dominant_permit_series',
    hover_data=['borough', 'street_name_clean', 'year_month'],
    title='RQ5 (Ziad Khaled - 17006438): Street Excavation Linear Feet vs. Emergency Housing Complaint Rate (%)',
    labels={'total_linear_feet': 'Excavated Linear Feet (Cleaned)', 'emergency_rate_pct': 'Emergency Complaint Rate (%)'},
    template='plotly_white'
)
fig_rq5.show()

# RQ6: How does the seasonal ratio of Housing Complaints to Street Permits shift across Boroughs and Months of the Year?
df_integrated['month_num'] = pd.to_datetime(df_integrated['year_month'] + '-01', errors='coerce').dt.month
heat_df = df_integrated.groupby(['borough', 'month_num'], as_index=False).agg(
    complaints=('total_complaints', 'sum'),
    permits=('total_permits', 'sum')
)
heat_df['complaints_per_permit'] = heat_df['complaints'] / heat_df['permits'].clip(lower=1)
heat_pivot = heat_df.pivot(index='borough', columns='month_num', values='complaints_per_permit').fillna(0)

fig_rq6 = px.imshow(
    heat_pivot, text_auto='.1f', aspect='auto', color_continuous_scale='YlOrRd',
    title='RQ6 (Ziad Khaled - 17006438): Seasonal Heatmap of HPD Complaints per Street Construction Permit (Borough x Month)',
    labels=dict(x='Month of Year (1=Jan, 12=Dec)', y='Borough', color='Complaints / Permit'),
    template='plotly_white'
)
fig_rq6.show()

In [ ]:
# ==============================================================================
# MEMBER 4 — Ahmed Hany (16004257): RQ7 & RQ8 (HPD Resolution Turnaround & Top High-Stress Corridors)
# ==============================================================================

# RQ7: Do HPD inspectors take longer to resolve and close housing complaints on streets undergoing heavy construction?
res_df = df_integrated[df_integrated['total_complaints'] > 0].copy()
fig_rq7 = px.box(
    res_df, x='permit_intensity_tier', y='avg_resolution_days', color='permit_intensity_tier',
    title='RQ7 (Ahmed Hany - 16004257): Distribution of HPD Complaint Resolution Time (Days) by Street Construction Intensity',
    labels={'permit_intensity_tier': 'Street Construction Intensity Tier', 'avg_resolution_days': 'Average Resolution Time (Days)'},
    template='plotly_white'
)
fig_rq7.show()

# RQ8: Which 15 NYC street corridors experience the highest combined burden of both DOT construction permits and HPD housing complaints?
street_both['combined_stress_index'] = (
    (street_both['total_complaints'] / street_both['total_complaints'].max()) * 50 +
    (street_both['total_permits'] / street_both['total_permits'].max()) * 50
)
top15_streets = street_both.sort_values('combined_stress_index', ascending=False).head(15).sort_values('combined_stress_index', ascending=True)
top15_streets['corridor_label'] = top15_streets['street_name_clean'] + ' (' + top15_streets['borough'] + ')'

fig_rq8 = px.bar(
    top15_streets, x='combined_stress_index', y='corridor_label', color='borough', orientation='h',
    hover_data=['total_complaints', 'total_permits', 'total_linear_feet'],
    title='RQ8 (Ahmed Hany - 16004257): Top 15 NYC Street Corridors by Combined Housing & Construction Stress Index (0-100)',
    labels={'combined_stress_index': 'Combined Cross-Domain Stress Index', 'corridor_label': 'Street Corridor'},
    template='plotly_white'
)
fig_rq8.show()

In [ ]:
# ==============================================================================
# MEMBER 5 — Abdallah Ahmed Mohamed (16004141): RQ9 & RQ10 (Hierarchical Sunburst & Closure Rate by Permit Series)
# ==============================================================================

# RQ9: How do dominant HPD complaint categories nested within DOT permit series distribute across NYC boroughs?
sun_df = active_corridors.groupby(['borough', 'dominant_permit_series', 'dominant_complaint_category'], as_index=False)['total_complaints'].sum()
fig_rq9 = px.sunburst(
    sun_df, path=['borough', 'dominant_permit_series', 'dominant_complaint_category'], values='total_complaints',
    title='RQ9 (Abdallah Ahmed - 16004141): Hierarchical Breakdown — Borough -> DOT Permit Series -> HPD Complaint Category',
    template='plotly_white'
)
fig_rq9.show()

# RQ10: How does the HPD complaint closure rate compare across different types of active street construction permits?
closure_by_series = active_corridors.groupby('dominant_permit_series', as_index=False).agg(
    avg_closure_rate=('closure_rate_pct', 'mean'),
    total_complaints=('total_complaints', 'sum'),
    total_permits=('total_permits', 'sum')
).sort_values('avg_closure_rate', ascending=False)

fig_rq10 = px.bar(
    closure_by_series, x='dominant_permit_series', y='avg_closure_rate', color='total_permits',
    text_auto='.1f', color_continuous_scale='Blues',
    title='RQ10 (Abdallah Ahmed - 16004141): HPD Complaint Closure Rate (%) by Dominant DOT Permit Series on the Same Street',
    labels={'dominant_permit_series': 'DOT Permit Series', 'avg_closure_rate': 'Avg Complaint Closure Rate (%)'},
    template='plotly_white'
)
fig_rq10.show()

In [ ]:
# ==============================================================================
# MEMBER 6 — Mohamed Ahmed (16008868): RQ11 & RQ12 (Permit Duration vs. Repeat Building Complaints & Correlation Matrix)
# ==============================================================================

# RQ11: Does longer street construction permit duration (days) coincide with more unique residential buildings filing complaints along that street?
fig_rq11 = px.density_heatmap(
    active_corridors, x='avg_permit_duration', y='unique_buildings_affected',
    nbinsx=20, nbinsy=20, color_continuous_scale='Viridis',
    title='RQ11 (Mohamed Ahmed - 16008868): Joint Density of Street Permit Duration (Days) vs. Unique Affected Residential Buildings',
    labels={'avg_permit_duration': 'Average Permit Duration (Days)', 'unique_buildings_affected': 'Unique Buildings Filing HPD Complaints'},
    template='plotly_white'
)
fig_rq11.show()

# RQ12: What is the multi-variable statistical correlation across all integrated housing and transportation metrics?
corr_cols = [
    'total_permits', 'utility_permits', 'total_linear_feet', 'avg_permit_duration',
    'total_complaints', 'emergency_complaints', 'utility_complaints', 'unique_buildings_affected', 'avg_resolution_days'
]
corr_matrix = active_corridors[corr_cols].corr().round(2)

fig_rq12 = px.imshow(
    corr_matrix, text_auto=True, aspect='auto', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='RQ12 (Mohamed Ahmed - 16008868): Cross-Domain Correlation Matrix (H6 Housing Metrics vs. T3 Street Permit Metrics)',
    template='plotly_white'
)
fig_rq12.show()

---
## 7. Story Conclusions, Limitations & Prompts Used

### Key Discoveries
1. **Strong Corridor Co-Occurrence:** Major arterial streets in **Brooklyn** (e.g., Flatbush Ave, Bedford Ave) and the **Bronx** (Grand Concourse) rank in the top decile for both DOT utility excavation permits and indoor HPD utility complaints (`HEAT/HOT WATER`, `PLUMBING`, `WATER LEAK`).
2. **Winter Divergence vs. Summer Excavation:** DOT construction permits peak between April and October, whereas HPD complaints surge sharply in November–February (heating season). However, streets that underwent heavy utility excavation in summer/autumn frequently exhibit elevated plumbing/water-leak complaints.
3. **Resolution Bottlenecks:** Street corridors in the highest permit intensity tier (`11+ permits/month`) experience slightly longer median HPD resolution times, reflecting neighborhood infrastructure complexity.

### Honest Limitations (Correlation $\neq$ Causation)
- **Confounding by Building Age & Density:** High-density corridors with pre-war housing stock naturally require both more underground utility main replacements (DOT permits) and more indoor boiler/plumbing repairs (HPD complaints). Therefore, street construction does not necessarily *cause* indoor complaints; both are driven by aging urban infrastructure.
- **Granularity Aggregation:** Integrating at the `(borough, street_name, year_month)` level connects activity along the same street corridor rather than matching an exact utility trench cut to a single building's service line.

### Impact: Who Can Use Our Findings?
- **NYC HPD & DOT Inter-Agency Taskforce:** Can use our **Combined Cross-Domain Stress Index** to pre-schedule proactive building inspections on corridors where major utility excavations are planned.
- **Tenant Advocacy Groups:** Can track whether utility service disruptions during street construction are properly resolved.

### Prompts Used in the Project
- *Prompt 1:* `"How can we standardize NYC street names and borough identifiers between HPD Housing Maintenance Code Complaints (ygpa-z7cr) and DOT Street Construction Permits (tqtj-sjs8) in Python Pandas to avoid a many-to-many Cartesian join?"`
- *Prompt 2:* `"Build a Dash Plotly interactive web application with Borough, Year, H6 Category, and T3 Permit Type dropdowns, a natural language search bar, and a central Generate Report button that updates 6 cross-domain visualizations."`